<a href="https://colab.research.google.com/github/Mona-Sbakhi/Hybrid_fpa-sda/blob/main/hybrid_fpa_sda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid FPA-SDA for ANN Optimization in Medical Imaging

## Mona Alsbakhi

**Author**: Mona Abdulfattah Alsbakhi
**Date**: November 8, 2025  
**Version**: 5.6

---

## Overview

This notebook implements a novel **Hybrid Flower Pollination and Seed Dispersal Algorithm (Hybrid FPA-SDA)** for optimizing Artificial Neural Networks in medical image classification tasks. This enhanced version includes **PCA and SGD as additional baseline algorithms** for comprehensive research validation.

### Key Features

- **Novel Hybrid Algorithm**: Sequential combination of FPA (global exploration) and SDA (local refinement)
- **Medical Imaging Application**: Chest X-ray pneumonia classification
- **Comprehensive Evaluation**: 30 independent runs, K-fold cross-validation, statistical testing
- **Six Algorithm Comparison**: Hybrid FPA-SDA, FPA, SDA,  **PCA**, and **SGD** (baselines)
- **Backup to Google Drive**

---

### Before Starting
- **In step 3**: Update username and key of kaggle


## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q opencv-python-headless scikit-posthocs kaggle

# Create project directory
!mkdir -p /content/hybrid_fpa_sda/src
!mkdir -p /content/hybrid_fpa_sda/results
!mkdir -p /content/hybrid_fpa_sda/data

# Change to project directory
%cd /content/hybrid_fpa_sda

## 1.1 Google Drive Integration

In [ ]:
# @title
# ============================================================================
# Google Drive Integration
# ============================================================================
# Run this cell to mount Google Drive and setup automatic backup

from google.colab import drive
import os
import shutil
from pathlib import Path

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive', force_remount=False)
print("✓ Google Drive mounted successfully!")

# Define paths
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/hybrid_fpa_sda_research'
LOCAL_PROJECT_PATH = '/content/hybrid_fpa_sda'
DATASET_PATH = '/content/hybrid_fpa_sda/data/chest_xray'

# Create project directory in Google Drive
os.makedirs(DRIVE_PROJECT_PATH, exist_ok=True)
print(f"✓ Project directory created: {DRIVE_PROJECT_PATH}")

# ============================================================================
# Function: Backup to Google Drive
# ============================================================================
def backup_to_drive(exclude_dataset=True):
    """
    Backup project files to Google Drive.

    Args:
        exclude_dataset: If True, skip dataset folder to save space and time
    """
    print("\n" + "="*70)
    print("BACKING UP TO GOOGLE DRIVE")
    print("="*70)

    # Directories to backup
    backup_dirs = ['src', 'results']

    for dir_name in backup_dirs:
        local_dir = os.path.join(LOCAL_PROJECT_PATH, dir_name)
        drive_dir = os.path.join(DRIVE_PROJECT_PATH, dir_name)

        if os.path.exists(local_dir):
            # Remove old backup and create new one
            if os.path.exists(drive_dir):
                shutil.rmtree(drive_dir)
                print(f"  ✓ Removed old backup: {dir_name}/")

            shutil.copytree(local_dir, drive_dir)
            print(f"  ✓ Backed up: {dir_name}/")

    # Backup dataset only if requested and exists
    if not exclude_dataset and os.path.exists(DATASET_PATH):
        drive_dataset = os.path.join(DRIVE_PROJECT_PATH, 'data')
        if os.path.exists(drive_dataset):
            shutil.rmtree(drive_dataset)
        shutil.copytree(os.path.join(LOCAL_PROJECT_PATH, 'data'), drive_dataset)
        print(f"  ✓ Backed up: data/")
    elif exclude_dataset:
        print(f"  ⊗ Skipped: data/ (excluded)")

    print("="*70)
    print("✓ Backup completed successfully!")
    print(f"Location: {DRIVE_PROJECT_PATH}")
    print("="*70)

# ============================================================================
# Function: Restore from Google Drive
# ============================================================================
def restore_from_drive(restore_dataset=False):
    """
    Restore project files from Google Drive.

    Args:
        restore_dataset: If True, restore dataset (only if not already present)
    """
    print("\n" + "="*70)
    print("RESTORING FROM GOOGLE DRIVE")
    print("="*70)

    # Check if backup exists
    if not os.path.exists(DRIVE_PROJECT_PATH):
        print("  ⚠ No backup found in Google Drive")
        return

    # Restore source code
    drive_src = os.path.join(DRIVE_PROJECT_PATH, 'src')
    if os.path.exists(drive_src):
        local_src = os.path.join(LOCAL_PROJECT_PATH, 'src')
        if os.path.exists(local_src):
            shutil.rmtree(local_src)
        shutil.copytree(drive_src, local_src)
        print(f"  ✓ Restored: src/")

    # Restore results
    drive_results = os.path.join(DRIVE_PROJECT_PATH, 'results')
    if os.path.exists(drive_results):
        local_results = os.path.join(LOCAL_PROJECT_PATH, 'results')
        if os.path.exists(local_results):
            shutil.rmtree(local_results)
        shutil.copytree(drive_results, local_results)
        print(f"  ✓ Restored: results/")

    # Restore dataset only if requested and not already present
    if restore_dataset:
        if not os.path.exists(DATASET_PATH):
            drive_data = os.path.join(DRIVE_PROJECT_PATH, 'data')
            if os.path.exists(drive_data):
                shutil.copytree(drive_data, os.path.join(LOCAL_PROJECT_PATH, 'data'))
                print(f"  ✓ Restored: data/")
            else:
                print(f"  ⚠ Dataset not found in backup")
        else:
            print(f"  ⊗ Skipped: data/ (already exists locally)")

    print("="*70)
    print("✓ Restore completed successfully!")
    print("="*70)

# ============================================================================
# Function: Auto-backup after experiments
# ============================================================================
def auto_backup_results():
    """Quick backup of results only (fast)"""
    print("\n" + "="*70)
    print("AUTO-BACKUP: Results Only")
    print("="*70)

    local_results = os.path.join(LOCAL_PROJECT_PATH, 'results')
    drive_results = os.path.join(DRIVE_PROJECT_PATH, 'results')

    if os.path.exists(local_results):
        if os.path.exists(drive_results):
            shutil.rmtree(drive_results)
        shutil.copytree(local_results, drive_results)
        print(f"  ✓ Results backed up to Google Drive")

    print("="*70)

# ============================================================================
# Initial Setup
# ============================================================================
print("\n" + "="*70)
print("GOOGLE DRIVE INTEGRATION READY")
print("="*70)
print("\nAvailable functions:")
print("  • backup_to_drive()           - Full backup (excludes dataset)")
print("  • backup_to_drive(False)      - Full backup (includes dataset)")
print("  • restore_from_drive()        - Restore code and results")
print("  • restore_from_drive(True)    - Restore everything (if dataset missing)")
print("  • auto_backup_results()       - Quick results backup")
print("\nRecommended workflow:")
print("  1. After each experiment: auto_backup_results()")
print("  2. At end of session: backup_to_drive()")
print("  3. At start of new session: restore_from_drive()")
print("="*70)

# Check if previous backup exists
if os.path.exists(os.path.join(DRIVE_PROJECT_PATH, 'results')):
    print("\n⚠️  Previous backup detected!")
    print("Run restore_from_drive() to load previous results")

## 2. Embed Source Code

The following cells use the `%%writefile` magic command to create the necessary Python modules in the `/content/hybrid_fpa_sda/src/` directory. **Run all these cells.**

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/base_optimizer.py
"""
Base Optimizer Class and Utility Functions
"""

import numpy as np
import math # FIX: Added for NumPy 2.0 compatibility
from typing import Callable, Tuple, Optional, Dict, List


class BaseOptimizer:
    """
    Base class for metaheuristic optimization algorithms.
    """

    def __init__(
        self,
        fitness_function: Callable[[np.ndarray], float],
        dim: int,
        pop_size: int = 25,
        max_iter: int = 100,
        bounds: Tuple[float, float] = (-1.0, 1.0),
        seed: Optional[int] = None
    ):
        self.fitness_function = fitness_function
        self.dim = dim
        self.pop_size = pop_size
        self.max_iter = max_iter
        self.bounds = bounds
        self.seed = seed

        if self.seed is not None:
            np.random.seed(self.seed)

        # Optimization state
        self.population = None
        self.fitness = None
        self.best_solution = None
        self.best_fitness = np.inf
        self.best_fitness_history = []

    def initialize_population(self):
        """Initialize population randomly within bounds."""
        lower, upper = self.bounds
        self.population = np.random.uniform(lower, upper, (self.pop_size, self.dim))

    def evaluate_population(self):
        """Evaluate the fitness of the entire population."""
        self.fitness = np.array([self.fitness_function(p) for p in self.population])

        # Update global best
        min_fitness_index = np.argmin(self.fitness)
        if self.fitness[min_fitness_index] < self.best_fitness:
            self.best_fitness = self.fitness[min_fitness_index]
            self.best_solution = self.population[min_fitness_index].copy()

    def record_iteration(self, iteration: int):
        """Record the best fitness for the current iteration."""
        self.best_fitness_history.append(self.best_fitness)

        if iteration % 10 == 0 or iteration == self.max_iter:
            print(f"Iteration {iteration}/{self.max_iter}: Best Fitness = {self.best_fitness:.6f}")

    def levy_flight(self, beta: float = 1.5) -> np.ndarray:
        """Generate a step size using Lévy flight distribution."""
        # Standard deviation for the numerator
        sigma_num = (math.gamma(1 + beta) * np.sin(np.pi * beta / 2)) # FIX: np.math.gamma -> math.gamma
        sigma_den = (math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2))) # FIX: np.math.gamma -> math.gamma
        sigma = (sigma_num / sigma_den) ** (1 / beta)

        # Generate u and v from normal distribution
        u = np.random.normal(0, sigma, self.dim)
        v = np.random.normal(0, 1, self.dim)

        # Lévy step
        step = u / (np.abs(v) ** (1 / beta))

        return step

    def clip_solution(self, solution: np.ndarray) -> np.ndarray:
        """Clip solution to stay within bounds."""
        lower, upper = self.bounds
        return np.clip(solution, lower, upper)

    def optimize(self) -> Tuple[np.ndarray, float, List[float]]:
        """Abstract method for running the optimization."""
        raise NotImplementedError("Subclasses must implement the optimize method")

    def get_convergence_data(self) -> Dict:
        """Return convergence history."""
        return {
            'history': self.best_fitness_history,
            'initial_fitness': self.best_fitness_history[0] if self.best_fitness_history else np.inf,
            'final_fitness': self.best_fitness
        }

    def get_algorithm_info(self) -> Dict:
        """Return algorithm configuration info."""
        return {
            'algorithm': self.__class__.__name__,
            'population_size': self.pop_size,
            'max_iterations': self.max_iter,
            'bounds': self.bounds,
            'dimension': self.dim
        }

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/fpa_optimizer.py
"""
Flower Pollination Algorithm (FPA) Implementation
"""

import numpy as np
from typing import Tuple, List, Optional, Callable
from base_optimizer import BaseOptimizer


class FlowerPollinationAlgorithm(BaseOptimizer):
    """
    Flower Pollination Algorithm (FPA) for global optimization.
    """

    def __init__(
        self,
        fitness_function: Callable[[np.ndarray], float],
        dim: int,
        pop_size: int = 25,
        max_iter: int = 100,
        bounds: Tuple[float, float] = (-1.0, 1.0),
        switch_probability: float = 0.8,
        seed: Optional[int] = None
    ):
        super().__init__(fitness_function, dim, pop_size, max_iter, bounds, seed)
        self.switch_probability = switch_probability

    def global_pollination(self, current_solution: np.ndarray, iteration: int) -> np.ndarray:
        """Global pollination (biotic and cross-pollination) using Lévy flights."""
        # Lévy flight step
        L = self.levy_flight()

        # Global pollination equation
        new_solution = current_solution + L * (self.best_solution - current_solution)

        return self.clip_solution(new_solution)

    def local_pollination(self, current_solution: np.ndarray) -> np.ndarray:
        """Local pollination (abiotic and self-pollination)."""
        # Choose two distinct solutions j and k randomly from the population
        j, k = np.random.choice(self.pop_size, 2, replace=False)

        # Epsilon (local random walk step size)
        epsilon = np.random.uniform(0, 1, self.dim)

        # Local pollination equation
        new_solution = current_solution + epsilon * (self.population[j] - self.population[k])

        return self.clip_solution(new_solution)

    def optimize(self) -> Tuple[np.ndarray, float, List[float]]:
        """Run the FPA optimization process."""
        self.initialize_population()
        self.evaluate_population()
        self.record_iteration(0)

        for iteration in range(1, self.max_iter + 1):
            new_population = np.zeros_like(self.population)

            for i in range(self.pop_size):
                # Decide between global and local pollination
                if np.random.rand() < self.switch_probability:
                    # Global pollination
                    new_solution = self.global_pollination(self.population[i], iteration)
                else:
                    # Local pollination
                    new_solution = self.local_pollination(self.population[i])

                # Evaluate new solution
                new_fitness = self.fitness_function(new_solution)

                # Greedy selection: replace if new solution is better
                if new_fitness < self.fitness[i]:
                    self.population[i] = new_solution
                    self.fitness[i] = new_fitness

                    # Update global best
                    if new_fitness < self.best_fitness:
                        self.best_fitness = new_fitness
                        self.best_solution = new_solution.copy()

            self.record_iteration(iteration)

        return self.best_solution, self.best_fitness, self.best_fitness_history

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/sda_optimizer.py
"""
Seed Dispersal Algorithm (SDA) Implementation
"""

import numpy as np
from typing import Tuple, List, Optional, Callable, Dict
from base_optimizer import BaseOptimizer


class SeedDispersalAlgorithm(BaseOptimizer):
    """
    Seed Dispersal Algorithm (SDA) for global optimization.
    Uses three dispersal mechanisms: global, local, and animal-mediated.
    """

    def __init__(
        self,
        fitness_function: Callable[[np.ndarray], float],
        dim: int,
        pop_size: int = 25,
        max_iter: int = 100,
        bounds: Tuple[float, float] = (-1.0, 1.0),
        dispersal_probs: Tuple[float, float, float] = (0.7, 0.15, 0.15), # Global, Local, Animal
        seed: Optional[int] = None
    ):
        super().__init__(fitness_function, dim, pop_size, max_iter, bounds, seed)
        self.dispersal_probs = dispersal_probs
        self.prob_cumsum = np.cumsum(dispersal_probs)

    def select_dispersal_mechanism(self) -> str:
        """Select dispersal mechanism based on probabilities."""
        r = np.random.rand()
        if r < self.prob_cumsum[0]:
            return 'global'
        elif r < self.prob_cumsum[1]:
            return 'local'
        else:
            return 'animal'

    def global_dispersal(self, current_seed: np.ndarray, iteration: int) -> np.ndarray:
        """Global dispersal (wind/water) using Lévy flights."""
        # Lévy flight step
        L = self.levy_flight()

        # Global dispersal equation
        new_seed = current_seed + L * (self.best_solution - current_seed)

        return self.clip_solution(new_seed)

    def local_dispersal(self, current_seed: np.ndarray, iteration: int) -> np.ndarray:
        """Local dispersal (gravity/rain) using random walk."""
        # Random step size
        step = np.random.uniform(-0.1, 0.1, self.dim)

        # Local dispersal equation
        new_seed = current_seed + step

        return self.clip_solution(new_seed)

    def animal_mediated_dispersal(self, current_seed: np.ndarray) -> np.ndarray:
        """Animal-mediated dispersal (guided search)."""
        # Choose a random seed from the population
        k = np.random.randint(0, self.pop_size)

        # Random step size
        epsilon = np.random.uniform(0, 1, self.dim)

        # Animal-mediated dispersal equation
        new_seed = current_seed + epsilon * (self.population[k] - current_seed)

        return self.clip_solution(new_seed)

    def optimize(self) -> Tuple[np.ndarray, float, List[float]]:
        """Run the SDA optimization process."""
        self.initialize_population()
        self.evaluate_population()
        self.record_iteration(0)

        for iteration in range(1, self.max_iter + 1):
            for i in range(self.pop_size):
                # Select and apply dispersal mechanism
                mechanism = self.select_dispersal_mechanism()

                if mechanism == 'global':
                    new_seed = self.global_dispersal(self.population[i], iteration)
                elif mechanism == 'local':
                    new_seed = self.local_dispersal(self.population[i], iteration)
                else:  # animal
                    new_seed = self.animal_mediated_dispersal(self.population[i])

                # Evaluate new solution
                new_fitness = self.fitness_function(new_seed)

                # Greedy selection: replace if new solution is better
                if new_fitness < self.fitness[i]:
                    self.population[i] = new_seed
                    self.fitness[i] = new_fitness

                    # Update global best
                    if new_fitness < self.best_fitness:
                        self.best_fitness = new_fitness
                        self.best_solution = new_seed.copy()

            self.record_iteration(iteration)

        return self.best_solution, self.best_fitness, self.best_fitness_history

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/hybrid_optimizer.py
"""
Hybrid FPA-SDA Algorithm Implementation
Novel sequential hybrid approach combining FPA and SDA
"""

import numpy as np
from typing import Tuple, List, Optional, Callable, Dict
from base_optimizer import BaseOptimizer
from fpa_optimizer import FlowerPollinationAlgorithm
from sda_optimizer import SeedDispersalAlgorithm


class HybridFPASDA(BaseOptimizer):
    """
    Hybrid FPA-SDA Algorithm for global optimization.

    This novel algorithm combines the strengths of both FPA and SDA in a
    sequential manner:
    - Phase 1: FPA performs global exploration using Lévy flights
    - Phase 2: SDA performs local refinement using multiple dispersal mechanisms

    The transition point between phases is configurable.
    """

    def __init__(
        self,
        fitness_function: Callable[[np.ndarray], float],
        dim: int,
        pop_size: int = 25,
        max_iter: int = 100,
        bounds: Tuple[float, float] = (-1.0, 1.0),
        transition_point: int = 50,
        fpa_switch_prob: float = 0.8,
        sda_dispersal_probs: Tuple[float, float, float] = (0.7, 0.15, 0.15),
        seed: Optional[int] = None
    ):
        """
        Initialize the Hybrid FPA-SDA Algorithm.
        """
        super().__init__(fitness_function, dim, pop_size, max_iter, bounds, seed)

        self.transition_point = transition_point
        self.fpa_switch_prob = fpa_switch_prob
        self.sda_dispersal_probs = sda_dispersal_probs

        # Validate transition point
        if transition_point <= 0 or transition_point >= max_iter:
            # FIX: Allow transition_point to be equal to max_iter if max_iter > 0
            if max_iter > 0 and transition_point == max_iter:
                pass # FPA only
            else:
                raise ValueError(
                    f"Transition point must be between 0 and {max_iter}, "
                    f"got {transition_point}"
                )

        # Initialize component algorithms (will be created during optimization)
        self.fpa = None
        self.sda = None

        # Track which phase is active
        self.current_phase = None
        self.phase_history = []

    def _initialize_fpa(self) -> FlowerPollinationAlgorithm:
        """Initialize FPA component with current state."""
        fpa = FlowerPollinationAlgorithm(
            fitness_function=self.fitness_function,
            dim=self.dim,
            pop_size=self.pop_size,
            max_iter=self.transition_point, # FPA runs up to transition point
            bounds=self.bounds,
            switch_probability=self.fpa_switch_prob,
            seed=None
        )
        return fpa

    def _initialize_sda(self) -> SeedDispersalAlgorithm:
        """Initialize SDA component with current state."""
        sda_iters = self.max_iter - self.transition_point
        sda = SeedDispersalAlgorithm(
            fitness_function=self.fitness_function,
            dim=self.dim,
            pop_size=self.pop_size,
            max_iter=sda_iters, # SDA runs for remaining iterations
            bounds=self.bounds,
            dispersal_probs=self.sda_dispersal_probs,
            seed=None
        )
        return sda

    def optimize(self) -> Tuple[np.ndarray, float, List[float]]:
        """Run the Hybrid FPA-SDA optimization."""
        print(f"\n{'='*60}")
        print(f"Starting Hybrid FPA-SDA Optimization")
        print(f"{'='*60}")

        # ===================================================================
        # PHASE 1: FPA for Global Exploration
        # ===================================================================
        print(f"PHASE 1: Flower Pollination Algorithm (Global Exploration)")

        self.current_phase = 'FPA'
        self.fpa = self._initialize_fpa()

        # Initialize population
        self.fpa.initialize_population()
        self.fpa.evaluate_population()
        self.fpa.record_iteration(0)

        # Copy initial state to hybrid optimizer
        self.population = self.fpa.population.copy()
        self.fitness = self.fpa.fitness.copy()
        self.best_solution = self.fpa.best_solution.copy()
        self.best_fitness = self.fpa.best_fitness
        self.record_iteration(0)

        # Run FPA phase
        for iteration in range(1, self.transition_point + 1):
            # FPA logic (copied from fpa_optimizer.py for self-contained execution)
            new_population = np.zeros_like(self.fpa.population)

            for i in range(self.pop_size):
                if np.random.rand() < self.fpa.switch_probability:
                    # Global pollination
                    L = self.fpa.levy_flight()
                    new_solution = self.fpa.population[i] + L * (self.fpa.best_solution - self.fpa.population[i])
                else:
                    # Local pollination
                    j, k = np.random.choice(self.pop_size, 2, replace=False)
                    epsilon = np.random.uniform(0, 1, self.dim)
                    new_solution = self.fpa.population[i] + epsilon * (self.fpa.population[j] - self.fpa.population[k])

                new_solution = self.fpa.clip_solution(new_solution)
                new_fitness = self.fitness_function(new_solution)

                if new_fitness < self.fpa.fitness[i]:
                    self.fpa.population[i] = new_solution
                    self.fpa.fitness[i] = new_fitness

                    if new_fitness < self.fpa.best_fitness:
                        self.fpa.best_fitness = new_fitness
                        self.fpa.best_solution = new_solution.copy()

            # Update hybrid optimizer state
            self.population = self.fpa.population.copy()
            self.fitness = self.fpa.fitness.copy()
            self.best_solution = self.fpa.best_solution.copy()
            self.best_fitness = self.fpa.best_fitness

            # Record iteration
            self.record_iteration(iteration)
            self.phase_history.append('FPA')

            # Print progress
            if iteration % 10 == 0:
                print(f"  FPA Iteration {iteration}/{self.transition_point}: "
                      f"Best Fitness = {self.best_fitness:.6f}")

        print(f"\nPhase 1 Complete: Best Fitness = {self.best_fitness:.6f}")

        # ===================================================================
        # PHASE 2: SDA for Local Refinement
        # ===================================================================
        sda_iterations = self.max_iter - self.transition_point
        if sda_iterations > 0:
            print(f"PHASE 2: Seed Dispersal Algorithm (Local Refinement)")

            self.current_phase = 'SDA'
            self.sda = self._initialize_sda()

            # Transfer population from FPA to SDA
            self.sda.population = self.population.copy()
            self.sda.fitness = self.fitness.copy()
            self.sda.best_solution = self.best_solution.copy()
            self.sda.best_fitness = self.best_fitness

            # Run SDA phase
            for sda_iter in range(1, sda_iterations + 1):
                global_iter = self.transition_point + sda_iter

                for i in range(self.pop_size):
                    # Select and apply dispersal mechanism
                    r = np.random.rand()
                    if r < self.sda.prob_cumsum[0]:
                        mechanism = 'global'
                        L = self.sda.levy_flight()
                        new_seed = self.sda.population[i] + L * (self.sda.best_solution - self.sda.population[i])
                    elif r < self.sda.prob_cumsum[1]:
                        mechanism = 'local'
                        step = np.random.uniform(-0.1, 0.1, self.dim)
                        new_seed = self.sda.population[i] + step
                    else:  # animal
                        mechanism = 'animal'
                        k = np.random.randint(0, self.pop_size)
                        epsilon = np.random.uniform(0, 1, self.dim)
                        new_seed = self.sda.population[i] + epsilon * (self.sda.population[k] - self.sda.population[i])

                    new_seed = self.sda.clip_solution(new_seed)
                    new_fitness = self.fitness_function(new_seed)

                    # Greedy selection
                    if new_fitness < self.sda.fitness[i]:
                        self.sda.population[i] = new_seed
                        self.sda.fitness[i] = new_fitness

                        if new_fitness < self.sda.best_fitness:
                            self.sda.best_fitness = new_fitness
                            self.sda.best_solution = new_seed.copy()

                # Update hybrid optimizer state
                self.population = self.sda.population.copy()
                self.fitness = self.sda.fitness.copy()
                self.best_solution = self.sda.best_solution.copy()
                self.best_fitness = self.sda.best_fitness

                # Record iteration
                self.record_iteration(global_iter)
                self.phase_history.append('SDA')

                # Print progress
                if sda_iter % 10 == 0:
                    print(f"  SDA Iteration {sda_iter}/{sda_iterations} "
                          f"(Global: {global_iter}/{self.max_iter}): "
                          f"Best Fitness = {self.best_fitness:.6f}")

            print(f"\nPhase 2 Complete: Best Fitness = {self.best_fitness:.6f}")

        print(f"{'='*60}")
        print(f"Hybrid FPA-SDA Optimization Complete")
        print(f"Final Best Fitness: {self.best_fitness:.6f}")
        print(f"{'='*60}\n")

        return self.best_solution, self.best_fitness, self.best_fitness_history

    def get_algorithm_info(self) -> Dict:
        """Return algorithm configuration info."""
        info = super().get_algorithm_info()
        info.update({
            'algorithm': 'Hybrid FPA-SDA Algorithm',
            'transition_point': self.transition_point,
            'phase_1': 'FPA (Global Exploration)',
            'phase_1_iterations': self.transition_point,
            'phase_2': 'SDA (Local Refinement)',
            'phase_2_iterations': self.max_iter - self.transition_point,
            'fpa_switch_probability': self.fpa_switch_prob,
            'sda_dispersal_probs': self.sda_dispersal_probs,
        })
        return info

    def get_phase_transition_analysis(self) -> Dict:
        """Analyze the performance at phase transition."""
        if len(self.best_fitness_history) < self.transition_point + 1:
            return {}

        fpa_final_fitness = self.best_fitness_history[self.transition_point]
        sda_initial_fitness = self.best_fitness_history[self.transition_point]
        sda_final_fitness = self.best_fitness_history[-1]

        fpa_improvement = self.best_fitness_history[0] - fpa_final_fitness
        sda_improvement = sda_initial_fitness - sda_final_fitness
        total_improvement = self.best_fitness_history[0] - sda_final_fitness

        return {
            'fpa_phase_improvement': fpa_improvement,
            'sda_phase_improvement': sda_improvement,
            'total_improvement': total_improvement,
            'fpa_contribution_ratio': fpa_improvement / total_improvement if total_improvement > 0 else 0,
            'sda_contribution_ratio': sda_improvement / total_improvement if total_improvement > 0 else 0,
            'transition_fitness': fpa_final_fitness
        }

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/pca_optimizer.py
"""
PCA-based Dimensionality Reduction Optimizer
"""

import numpy as np
from typing import Callable, Tuple, List, Optional
from sklearn.decomposition import PCA
from base_optimizer import BaseOptimizer


class PCAOptimizer(BaseOptimizer):
    """
    PCA-based optimizer that reduces dimensionality before training.
    This serves as a baseline to compare feature reduction approaches.
    """

    def __init__(
        self,
        fitness_function: Callable[[np.ndarray], float],
        dim: int,
        pop_size: int = 25,
        max_iter: int = 100,
        bounds: Tuple[float, float] = (-1.0, 1.0),
        n_components: int = 100,
        seed: Optional[int] = None
    ):
        super().__init__(fitness_function, dim, pop_size, max_iter, bounds, seed)
        self.n_components = min(n_components, dim)  # Ensure valid component count

    def optimize(self) -> Tuple[np.ndarray, float, List[float]]:
        """
        Run PCA-based optimization.

        This is a wrapper that performs random search in the original space.
        The actual PCA transformation should be applied to input data before
        creating the fitness function.
        """
        print(f"\nStarting PCA Optimizer (Random Search Baseline)")
        print(f"Population Size: {self.pop_size}, Max Iterations: {self.max_iter}")
        print(f"Dimension: {self.dim}, Bounds: {self.bounds}\n")

        # Initialize population
        self.initialize_population()
        self.evaluate_population()
        self.record_iteration(0)

        # Random search iterations
        for iteration in range(1, self.max_iter + 1):
            # Generate new random solutions
            lower, upper = self.bounds
            new_population = np.random.uniform(lower, upper, (self.pop_size, self.dim))

            # Evaluate new population
            new_fitness = np.array([self.fitness_function(p) for p in new_population])

            # Update population with better solutions
            for i in range(self.pop_size):
                if new_fitness[i] < self.fitness[i]:
                    self.population[i] = new_population[i].copy()
                    self.fitness[i] = new_fitness[i]

                    # Update global best
                    if new_fitness[i] < self.best_fitness:
                        self.best_fitness = new_fitness[i]
                        self.best_solution = new_population[i].copy()

            self.record_iteration(iteration)

        print(f"\nOptimization Complete!")
        print(f"Best Fitness: {self.best_fitness:.6f}\n")

        return self.best_solution, self.best_fitness, self.best_fitness_history


In [ ]:
%%writefile /content/hybrid_fpa_sda/src/sgd_optimizer.py
"""
Stochastic Gradient Descent (SGD) Optimizer
"""

import numpy as np
from typing import Callable, Tuple, List, Optional
from base_optimizer import BaseOptimizer


class SGDOptimizer(BaseOptimizer):
    """
    Stochastic Gradient Descent optimizer using numerical gradients.
    Serves as a gradient-based baseline for comparison.
    """

    def __init__(
        self,
        fitness_function: Callable[[np.ndarray], float],
        dim: int,
        pop_size: int = 1,  # SGD typically uses single solution
        max_iter: int = 100,
        bounds: Tuple[float, float] = (-1.0, 1.0),
        learning_rate: float = 0.001,
        momentum: float = 0.9,
        epsilon: float = 1e-7,
        seed: Optional[int] = None
    ):
        super().__init__(fitness_function, dim, pop_size, max_iter, bounds, seed)
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.epsilon = epsilon
        self.velocity = None

    def compute_numerical_gradient(self, params: np.ndarray, epsilon: float = 1e-5) -> np.ndarray:
        """
        Compute numerical gradient using finite differences.
        Uses a subset of parameters for efficiency.
        """
        gradient = np.zeros_like(params)

        # Sample a subset of dimensions for gradient computation (for efficiency)
        sample_size = min(1000, self.dim)
        indices = np.random.choice(self.dim, sample_size, replace=False)

        f_x = self.fitness_function(params)

        for idx in indices:
            params_plus = params.copy()
            params_plus[idx] += epsilon
            f_x_plus = self.fitness_function(params_plus)
            gradient[idx] = (f_x_plus - f_x) / epsilon

        return gradient

    def optimize(self) -> Tuple[np.ndarray, float, List[float]]:
        """
        Run SGD optimization with momentum.
        """
        print(f"\nStarting SGD Optimizer")
        print(f"Learning Rate: {self.learning_rate}, Momentum: {self.momentum}")
        print(f"Max Iterations: {self.max_iter}, Dimension: {self.dim}\n")

        # Initialize solution randomly
        lower, upper = self.bounds
        self.best_solution = np.random.uniform(lower, upper, self.dim)
        self.velocity = np.zeros(self.dim)

        # Evaluate initial solution
        self.best_fitness = self.fitness_function(self.best_solution)
        self.best_fitness_history.append(self.best_fitness)

        print(f"Iteration 0/{self.max_iter}: Best Fitness = {self.best_fitness:.6f}")

        # SGD iterations
        for iteration in range(1, self.max_iter + 1):
            # Compute numerical gradient
            gradient = self.compute_numerical_gradient(self.best_solution, self.epsilon)

            # Update velocity with momentum
            self.velocity = self.momentum * self.velocity - self.learning_rate * gradient

            # Update parameters
            new_solution = self.best_solution + self.velocity

            # Clip to bounds
            new_solution = self.clip_solution(new_solution)

            # Evaluate new solution
            new_fitness = self.fitness_function(new_solution)

            # Update if better
            if new_fitness < self.best_fitness:
                self.best_solution = new_solution.copy()
                self.best_fitness = new_fitness

            self.best_fitness_history.append(self.best_fitness)

            if iteration % 10 == 0 or iteration == self.max_iter:
                print(f"Iteration {iteration}/{self.max_iter}: Best Fitness = {self.best_fitness:.6f}")

        print(f"\nOptimization Complete!")
        print(f"Best Fitness: {self.best_fitness:.6f}\n")

        return self.best_solution, self.best_fitness, self.best_fitness_history


In [ ]:
%%writefile /content/hybrid_fpa_sda/src/ann_model.py
"""
Artificial Neural Network (ANN) Model Implementation
"""

import numpy as np
from typing import List, Optional, Tuple
from scipy.special import softmax


class MLP:
    """
    Multi-Layer Perceptron (MLP) for classification.
    Uses ReLU activation for hidden layers and Softmax for output.
    """

    def __init__(
        self,
        input_size: int = 22500, # Default for 150x150 image
        hidden_sizes: List[int] = [128, 64, 32],
        output_size: int = 2,
        dropout_rate: float = 0.3,
        l2_lambda: float = 0.001,
        seed: Optional[int] = None
    ):
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.output_size = output_size
        self.dropout_rate = dropout_rate
        self.l2_lambda = l2_lambda
        self.seed = seed

        if self.seed is not None:
            np.random.seed(self.seed)

        self.layer_sizes = [input_size] + hidden_sizes + [output_size]
        self.weights = []
        self.biases = []
        self._initialize_parameters()

    def _initialize_parameters(self):
        """Initialize weights and biases using Xavier/He initialization."""
        self.weights = []
        self.biases = []

        for i in range(len(self.layer_sizes) - 1):
            in_dim = self.layer_sizes[i]
            out_dim = self.layer_sizes[i+1]

            # He initialization for ReLU
            scale = np.sqrt(2 / in_dim)
            W = np.random.randn(in_dim, out_dim) * scale
            b = np.zeros(out_dim)

            self.weights.append(W)
            self.biases.append(b)

    @property
    def total_params(self) -> int:
        """Calculate the total number of parameters in the network."""
        count = 0
        for W in self.weights:
            count += W.size
        for b in self.biases:
            count += b.size
        return count

    def get_parameters(self) -> np.ndarray:
        """Concatenate all weights and biases into a single vector."""
        params = []
        for W in self.weights:
            params.append(W.flatten())
        for b in self.biases:
            params.append(b.flatten())
        return np.concatenate(params)

    def set_parameters(self, params: np.ndarray):
        """Set weights and biases from a single parameter vector."""
        start = 0

        # Set Weights
        for i in range(len(self.weights)):
            size = self.weights[i].size
            self.weights[i] = params[start:start+size].reshape(self.weights[i].shape)
            start += size

        # Set Biases
        for i in range(len(self.biases)):
            size = self.biases[i].size
            self.biases[i] = params[start:start+size].reshape(self.biases[i].shape)
            start += size

    def forward(self, X: np.ndarray) -> np.ndarray:
        """Perform forward pass to get output probabilities."""
        A = X

        for i in range(len(self.weights)):
            Z = A @ self.weights[i] + self.biases[i]

            if i < len(self.weights) - 1:
                # ReLU activation for hidden layers
                A = np.maximum(0, Z)

                # Dropout (only during training, but metaheuristics don't use this path)
                # A = self._apply_dropout(A)
            else:
                # Softmax for output layer
                A = softmax(Z, axis=1)

        return A

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities."""
        return self.forward(X)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels."""
        probabilities = self.predict_proba(X)
        return np.argmax(probabilities, axis=1)

    def compute_loss(self, X: np.ndarray, y: np.ndarray, class_weights: Optional[np.ndarray] = None) -> float:
        """Compute the weighted categorical cross-entropy loss + L2 regularization."""
        m = X.shape[0]

        # Forward pass to get probabilities
        y_hat = self.forward(X)

        # Clip probabilities to avoid log(0)
        y_hat = np.clip(y_hat, 1e-12, 1.0 - 1e-12)

        # Convert y to one-hot encoding
        y_one_hot = np.zeros_like(y_hat)
        y_one_hot[np.arange(m), y] = 1

        # Categorical Cross-Entropy Loss
        if class_weights is not None:
            # Apply class weights
            weights = class_weights[y]
            log_likelihood = -np.log(y_hat[np.arange(m), y])
            cross_entropy_loss = np.sum(weights * log_likelihood) / m
        else:
            cross_entropy_loss = -np.sum(y_one_hot * np.log(y_hat)) / m

        # L2 Regularization Term
        l2_reg_term = 0
        for W in self.weights:
            l2_reg_term += np.sum(W**2)
        l2_reg_term = (self.l2_lambda / (2 * m)) * l2_reg_term

        # Total Loss
        total_loss = cross_entropy_loss + l2_reg_term

        return total_loss

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/data_handler.py
"""
Data Handler for Medical Image Classification
Preprocessing pipeline for Chest X-ray Pneumonia Dataset
"""

import numpy as np
from typing import Tuple, Optional, Dict, List
from pathlib import Path
import cv2
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight


class MedicalImageDataHandler:
    """
    Data handler for medical image classification tasks.
    """

    def __init__(
        self,
        image_size: Tuple[int, int] = (150, 150),
        normalize: bool = True,
        seed: Optional[int] = None
    ):
        self.image_size = image_size
        self.normalize = normalize
        self.seed = seed

        if seed is not None:
            np.random.seed(seed)

        # Data storage
        self.X_train = None
        self.y_train = None
        self.X_val = None
        self.y_val = None
        self.X_test = None
        self.y_test = None

        # Class information
        self.class_names = ['NORMAL', 'PNEUMONIA']
        self.num_classes = len(self.class_names)
        self.class_weights = None

    def load_images_from_directory(
        self,
        data_dir: str,
        max_samples: Optional[int] = None
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Load images from directory structure."""
        data_path = Path(data_dir)
        images = []
        labels = []

        for class_idx, class_name in enumerate(self.class_names):
            class_dir = data_path / class_name

            if not class_dir.exists():
                print(f"Warning: Directory {class_dir} does not exist")
                continue

            image_files = list(class_dir.glob('*.jpeg')) + \
                         list(class_dir.glob('*.jpg')) + \
                         list(class_dir.glob('*.png'))

            if max_samples is not None:
                # Calculate max samples per class to maintain balance
                max_per_class = max_samples // self.num_classes
                image_files = image_files[:max_per_class]

            print(f"Loading {len(image_files)} images from {class_name}...")

            for img_path in image_files:
                img = self._load_and_preprocess_image(str(img_path))
                if img is not None:
                    images.append(img)
                    labels.append(class_idx)

        images = np.array(images)
        labels = np.array(labels)

        print(f"Loaded {len(images)} images total")
        print(f"Class distribution: {np.bincount(labels)}")

        return images, labels

    def _load_and_preprocess_image(self, image_path: str) -> Optional[np.ndarray]:
        """Load and preprocess a single image."""
        try:
            # Load image in grayscale
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                # FIX: Handle corrupted/unreadable files gracefully
                # print(f"Warning: Failed to load {image_path}")
                return None

            # Resize to target size
            img = cv2.resize(img, self.image_size, interpolation=cv2.INTER_AREA)

            # Normalize to [0, 1]
            if self.normalize:
                img = img.astype(np.float32) / 255.0

            # Flatten to 1D vector
            img = img.flatten()

            return img

        except Exception as e:
            # print(f"Error loading {image_path}: {e}")
            return None

    def augment_image(self, image: np.ndarray) -> np.ndarray:
        """Apply data augmentation to an image."""
        # Reshape to 2D for augmentation
        img_2d = image.reshape(self.image_size)

        # Random rotation
        if np.random.rand() < 0.5:
            angle = np.random.uniform(-15, 15)
            center = (self.image_size[0] // 2, self.image_size[1] // 2)
            rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
            img_2d = cv2.warpAffine(img_2d, rotation_matrix, self.image_size)

        # Horizontal flip
        if np.random.rand() < 0.5:
            img_2d = cv2.flip(img_2d, 1)

        # Random zoom
        if np.random.rand() < 0.5:
            zoom_factor = np.random.uniform(0.9, 1.1)
            h, w = self.image_size
            new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)

            img_2d = cv2.resize(img_2d, (new_w, new_h))

            # Crop or pad to original size
            if zoom_factor > 1:
                # Crop center
                start_h = (new_h - h) // 2
                start_w = (new_w - w) // 2
                img_2d = img_2d[start_h:start_h+h, start_w:start_w+w]
            else:
                # Pad
                pad_h = (h - new_h) // 2
                pad_w = (w - new_w) // 2
                img_2d = cv2.copyMakeBorder(
                    img_2d, pad_h, h-new_h-pad_h, pad_w, w-new_w-pad_w,
                    cv2.BORDER_CONSTANT, value=0
                )

        # Flatten back to 1D
        return img_2d.flatten()

    def augment_dataset(
        self,
        X: np.ndarray,
        y: np.ndarray,
        augmentation_factor: int = 2
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Augment entire dataset."""
        X_augmented = [X]
        y_augmented = [y]

        for _ in range(augmentation_factor - 1):
            X_aug = np.array([self.augment_image(img) for img in X])
            X_augmented.append(X_aug)
            y_augmented.append(y)

        X_augmented = np.vstack(X_augmented)
        y_augmented = np.concatenate(y_augmented)

        # Shuffle
        indices = np.random.permutation(len(X_augmented))
        X_augmented = X_augmented[indices]
        y_augmented = y_augmented[indices]

        return X_augmented, y_augmented

    def compute_class_weights(self, y: np.ndarray) -> np.ndarray:
        """Compute class weights for imbalanced dataset."""
        class_weights = compute_class_weight(
            'balanced',
            classes=np.unique(y),
            y=y
        )

        self.class_weights = class_weights

        print(f"Class weights computed: {class_weights}")
        print(f"  NORMAL (0): {class_weights[0]:.4f}")
        print(f"  PNEUMONIA (1): {class_weights[1]:.4f}")

        return class_weights

    def split_data(
        self,
        X: np.ndarray,
        y: np.ndarray,
        test_size: float = 0.2,
        val_size: float = 0.2
    ) -> None:
        """Split data into train, validation, and test sets."""
        # First split: train+val and test
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y,
            test_size=test_size,
            stratify=y,
            random_state=self.seed
        )

        # Second split: train and val
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval,
            test_size=val_size, # FIX: Use val_size directly
            stratify=y_trainval,
            random_state=self.seed
        )

        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test

        print(f"\nData split complete:")
        print(f"  Training set: {len(X_train)} samples")
        print(f"  Validation set: {len(X_val)} samples")
        print(f"  Test set: {len(X_test)} samples")
        print(f"\nClass distribution:")
        print(f"  Train - NORMAL: {np.sum(y_train==0)}, PNEUMONIA: {np.sum(y_train==1)}")
        print(f"  Val   - NORMAL: {np.sum(y_val==0)}, PNEUMONIA: {np.sum(y_val==1)}")
        print(f"  Test  - NORMAL: {np.sum(y_test==0)}, PNEUMONIA: {np.sum(y_test==1)}")

    def get_kfold_splits(
        self,
        X: np.ndarray,
        y: np.ndarray,
        n_splits: int = 5
    ) -> StratifiedKFold:
        """Create K-fold cross-validation splits."""
        skf = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=self.seed
        )

        return skf

    def add_noise(
        self,
        X: np.ndarray,
        noise_level: float = 0.1
    ) -> np.ndarray:
        """Add Gaussian noise to images for robustness testing."""
        noise = np.random.normal(0, noise_level, X.shape)
        X_noisy = X + noise

        # Clip to valid range
        if self.normalize:
            X_noisy = np.clip(X_noisy, 0, 1)
        else:
            X_noisy = np.clip(X_noisy, 0, 255)

        return X_noisy

    def get_data_info(self) -> Dict:
        """Get information about the loaded dataset."""
        info = {
            'image_size': self.image_size,
            'feature_dimension': self.image_size[0] * self.image_size[1],
            'num_classes': self.num_classes,
            'class_names': self.class_names,
            'normalized': self.normalize
        }

        if self.X_train is not None:
            info.update({
                'train_samples': len(self.X_train),
                'val_samples': len(self.X_val),
                'test_samples': len(self.X_test),
                'train_class_dist': np.bincount(self.y_train).tolist(),
                'val_class_dist': np.bincount(self.y_val).tolist(),
                'test_class_dist': np.bincount(self.y_test).tolist()
            })

        if self.class_weights is not None:
            info['class_weights'] = self.class_weights.tolist()

        return info

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/evaluation.py
"""
Evaluation Framework for Model Performance Assessment
"""

import numpy as np
from typing import Dict, List, Tuple, Optional
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, matthews_corrcoef
)
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import seaborn as sns


class ModelEvaluator:
    """
    Comprehensive evaluation framework for classification models.
    """

    def __init__(self):
        """Initialize the evaluator."""
        self.results = {}

    def calculate_metrics(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        y_proba: Optional[np.ndarray] = None
    ) -> Dict[str, float]:
        """Calculate comprehensive classification metrics."""
        # Basic metrics
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='binary', zero_division=0)
        recall = recall_score(y_true, y_pred, average='binary', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='binary', zero_division=0)
        mcc = matthews_corrcoef(y_true, y_pred)

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)

        # Calculate specificity (True Negative Rate)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        else:
            specificity = 0

        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'specificity': specificity,
            'f1_score': f1,
            'mcc': mcc,
            'confusion_matrix': cm
        }

        return metrics

    def calculate_convergence_metrics(
        self,
        fitness_history: List[float],
        threshold: float = 0.95
    ) -> Dict[str, float]:
        """Calculate convergence-related metrics."""
        if len(fitness_history) == 0:
            return {}

        final_fitness = fitness_history[-1]
        initial_fitness = fitness_history[0]

        # Find convergence iteration (95% of final fitness)
        convergence_iter = -1
        target_fitness = final_fitness / threshold if final_fitness != 0 else 0

        for i, fitness in enumerate(fitness_history):
            if fitness <= target_fitness:
                convergence_iter = i + 1
                break

        if convergence_iter == -1:
            convergence_iter = len(fitness_history)

        # Calculate improvement
        improvement = initial_fitness - final_fitness
        improvement_pct = (improvement / initial_fitness * 100) if initial_fitness != 0 else 0

        # Calculate stability (std of last 10% iterations)
        last_10pct = int(len(fitness_history) * 0.1)
        stability_window = fitness_history[-last_10pct:] if last_10pct > 0 else fitness_history
        stability_std = np.std(stability_window)

        metrics = {
            'final_fitness': final_fitness,
            'initial_fitness': initial_fitness,
            'improvement': improvement,
            'improvement_percentage': improvement_pct,
            'convergence_iteration': convergence_iter,
            'convergence_speed': convergence_iter / len(fitness_history),
            'stability_std': stability_std
        }

        return metrics

    def aggregate_multiple_runs(
        self,
        results_list: List[Dict[str, float]]
    ) -> Dict[str, Dict[str, float]]:
        """Aggregate results from multiple independent runs."""
        if not results_list:
            return {}

        # Extract metric names
        metric_names = [k for k in results_list[0].keys() if k != 'confusion_matrix']

        aggregated = {}
        for metric in metric_names:
            values = [r[metric] for r in results_list if metric in r]

            if len(values) > 0:
                aggregated[metric] = {
                    'mean': np.mean(values),
                    'std': np.std(values),
                    'min': np.min(values),
                    'max': np.max(values),
                    'median': np.median(values),
                    'values': values
                }

        return aggregated

    def friedman_test(
        self,
        results_dict: Dict[str, List[float]],
        metric_name: str = 'accuracy'
    ) -> Tuple[float, float]:
        """Perform Friedman test for multiple algorithm comparison."""
        # Prepare data for Friedman test
        data_arrays = [np.array(values) for values in results_dict.values()]

        # Ensure all arrays have same length
        min_length = min(len(arr) for arr in data_arrays)
        data_arrays = [arr[:min_length] for arr in data_arrays]

        # Perform Friedman test
        statistic, p_value = friedmanchisquare(*data_arrays)

        print(f"\nFriedman Test Results for {metric_name}:")
        print(f"  Statistic: {statistic:.4f}")
        print(f"  P-value: {p_value:.6f}")

        if p_value < 0.05:
            print(f"  Result: Significant differences detected (p < 0.05)")
        else:
            print(f"  Result: No significant differences (p >= 0.05)")

        return statistic, p_value

    def nemenyi_test(
        self,
        results_dict: Dict[str, List[float]],
        metric_name: str = 'accuracy'
    ) -> np.ndarray:
        """Perform Nemenyi post-hoc test for pairwise comparisons."""
        # Prepare data matrix (algorithms x runs)
        algorithm_names = list(results_dict.keys())
        data_arrays = [np.array(results_dict[name]) for name in algorithm_names]

        # Ensure all arrays have same length
        min_length = min(len(arr) for arr in data_arrays)
        data_matrix = np.array([arr[:min_length] for arr in data_arrays]).T

        # Perform Nemenyi test
        p_values = sp.posthoc_nemenyi_friedman(data_matrix)
        p_values.index = algorithm_names
        p_values.columns = algorithm_names

        print(f"\nNemenyi Post-hoc Test Results for {metric_name}:")
        print(p_values.to_string())

        return p_values

    def plot_convergence_curves(
        self,
        convergence_data: Dict[str, List[float]],
        title: str = "Convergence Comparison",
        save_path: Optional[str] = None
    ) -> None:
        """Plot convergence curves for multiple algorithms."""
        plt.figure(figsize=(12, 7))

        for algo_name, fitness_history in convergence_data.items():
            iterations = range(len(fitness_history))
            plt.plot(iterations, fitness_history, label=algo_name, linewidth=2)

        plt.xlabel('Iteration', fontsize=12)
        plt.ylabel('Best Fitness (Loss)', fontsize=12)
        plt.title(title, fontsize=14, fontweight='bold')
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Convergence plot saved to {save_path}")

        plt.show()

    def plot_metrics_comparison(
        self,
        aggregated_results: Dict[str, Dict[str, Dict]],
        metrics: List[str] = ['accuracy', 'precision', 'recall', 'f1_score', 'mcc'],
        save_path: Optional[str] = None
    ) -> None:
        """Plot bar chart comparing metrics across algorithms."""
        algorithm_names = list(aggregated_results.keys())
        n_metrics = len(metrics)

        fig, axes = plt.subplots(1, n_metrics, figsize=(4*n_metrics, 5))
        if n_metrics == 1:
            axes = [axes]

        for idx, metric in enumerate(metrics):
            means = []
            stds = []

            for algo in algorithm_names:
                if metric in aggregated_results[algo]:
                    means.append(aggregated_results[algo][metric]['mean'])
                    stds.append(aggregated_results[algo][metric]['std'])
                else:
                    means.append(0)
                    stds.append(0)

            x_pos = np.arange(len(algorithm_names))
            ax = axes[idx]
            ax.bar(x_pos, means, yerr=stds, align='center', alpha=0.7, capsize=5)
            ax.set_ylabel('Mean Value')
            ax.set_xticks(x_pos)
            ax.set_xticklabels(algorithm_names, rotation=45, ha="right")
            ax.set_title(metric.replace('_', ' ').title())
            ax.yaxis.grid(True)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Metrics comparison plot saved to {save_path}")

        plt.show()

In [ ]:
%%writefile /content/hybrid_fpa_sda/src/experiment_manager.py
"""
Experiment Manager for Metaheuristic ANN Optimization
"""

import numpy as np
from typing import Dict, List, Optional, Callable, Tuple
import time
import pickle
from pathlib import Path

from ann_model import MLP
from fpa_optimizer import FlowerPollinationAlgorithm
from sda_optimizer import SeedDispersalAlgorithm
from hybrid_optimizer import HybridFPASDA
from pca_optimizer import PCAOptimizer
from sgd_optimizer import SGDOptimizer
from evaluation import ModelEvaluator


class ExperimentManager:
    """
    Manages experiments for comparing optimization algorithms.
    """

    def __init__(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        X_test: np.ndarray,
        y_test: np.ndarray,
        class_weights: Optional[np.ndarray] = None,
        results_dir: str = './results'
    ):
        self.X_train = X_train
        self.y_train = y_train
        self.X_val = X_val
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        self.class_weights = class_weights

        # Create results directory
        self.results_dir = Path(results_dir)
        self.results_dir.mkdir(parents=True, exist_ok=True)

        # Initialize evaluator
        self.evaluator = ModelEvaluator()

        # Storage for results
        self.all_results = {}

    def create_fitness_function(
        self,
        model: MLP,
        X: np.ndarray,
        y: np.ndarray
    ) -> Callable[[np.ndarray], float]:
        """Create fitness function for optimization algorithms."""
        def fitness_function(params: np.ndarray) -> float:
            try:
                # Set model parameters
                model.set_parameters(params)

                # Compute loss
                loss = model.compute_loss(X, y, self.class_weights)

                return loss
            except Exception as e:
                # print(f"Error in fitness evaluation: {e}") # Suppress for cleaner output
                return np.inf

        return fitness_function

    def train_with_metaheuristic(
        self,
        optimizer_class,
        optimizer_params: Dict,
        model: MLP,
        run_id: int = 0
    ) -> Tuple[MLP, Dict]:
        """Train model using a metaheuristic optimizer."""
        print(f"\n{'='*60}")
        print(f"Training with {optimizer_class.__name__} (Run {run_id + 1})")
        print(f"{'='*60}")

        start_time = time.time()

        # Create fitness function
        fitness_func = self.create_fitness_function(model, self.X_val, self.y_val)

        # Initialize optimizer
        optimizer = optimizer_class(
            fitness_function=fitness_func,
            dim=model.total_params,
            **optimizer_params
        )

        # Run optimization
        best_params, best_fitness, fitness_history = optimizer.optimize()

        training_time = time.time() - start_time

        # Set best parameters to model
        model.set_parameters(best_params)

        # Get convergence data
        convergence_data = optimizer.get_convergence_data()

        training_info = {
            'best_fitness': best_fitness,
            'fitness_history': fitness_history,
            'training_time': training_time,
            'convergence_data': convergence_data,
            'algorithm_info': optimizer.get_algorithm_info()
        }

        print(f"\nTraining completed in {training_time:.2f} seconds")
        print(f"Final fitness: {best_fitness:.6f}")

        return model, training_info

    def run_experiment(
        self,
        algorithm_name: str,
        n_runs: int = 30,
        adam_trainer_func: Optional[Callable] = None,
        **kwargs
    ) -> Dict:
        """
        Run multiple independent experiments for an algorithm.

        Args:
            algorithm_name: Name of algorithm ('FPA', 'SDA', 'Hybrid', 'PCA', 'SGD')
            n_runs: Number of independent runs
            **kwargs: Additional parameters for the algorithm

        Returns:
            Dictionary containing aggregated results
        """
        print(f"\n{'#'*60}")
        print(f"# Running {n_runs} experiments with {algorithm_name}")
        print(f"{'#'*60}\n")

        run_results = []
        convergence_histories = []

        for run_id in range(n_runs):
            # Create fresh model for each run
            # FIX: Extract input_size and hidden_sizes from kwargs if provided
            input_size = kwargs.get('input_size', 22500)
            hidden_sizes = kwargs.get('hidden_sizes', [128, 64, 32])
            model = MLP(input_size=input_size, hidden_sizes=hidden_sizes, seed=run_id)

            # Remove input_size and hidden_sizes from kwargs before passing to optimizer
            optimizer_kwargs = {k: v for k, v in kwargs.items() if k not in ['input_size', 'hidden_sizes']}

            # Train based on algorithm
            if algorithm_name == 'FPA':
                trained_model, training_info = self.train_with_metaheuristic(
                    FlowerPollinationAlgorithm,
                    optimizer_kwargs,
                    model,
                    run_id
                )
            elif algorithm_name == 'SDA':
                trained_model, training_info = self.train_with_metaheuristic(
                    SeedDispersalAlgorithm,
                    optimizer_kwargs,
                    model,
                    run_id
                )
            elif algorithm_name == 'Hybrid':
                trained_model, training_info = self.train_with_metaheuristic(
                    HybridFPASDA,
                    optimizer_kwargs,
                    model,
                    run_id
                )
            elif algorithm_name == 'SGD':
                trained_model, training_info = self.train_with_metaheuristic(
                    SGDOptimizer,
                    optimizer_kwargs,
                    model,
                    run_id
                )
            elif algorithm_name == 'PCA':
                trained_model, training_info = self.train_with_metaheuristic(
                    PCAOptimizer,
                    optimizer_kwargs,
                    model,
                    run_id
                )
            else:
                raise ValueError(f"Unknown algorithm: {algorithm_name}")

            # Evaluate on test set
            y_pred = trained_model.predict(self.X_test)
            y_proba = trained_model.predict_proba(self.X_test)

            metrics = self.evaluator.calculate_metrics(
                self.y_test, y_pred, y_proba
            )

            # Add convergence metrics
            conv_metrics = self.evaluator.calculate_convergence_metrics(
                training_info['fitness_history']
            )
            metrics.update(conv_metrics)

            # ✨ NEW: Add training time to metrics
            metrics['training_time'] = training_info['training_time']

            # Store results
            run_results.append(metrics)
            convergence_histories.append(training_info['fitness_history'])

            print(f"\nRun {run_id + 1} Test Results:")
            print(f"  Accuracy: {metrics['accuracy']:.4f}")
            print(f"  F1-Score: {metrics['f1_score']:.4f}")
            print(f"  MCC: {metrics['mcc']:.4f}")
            print(f"  Training Time: {metrics['training_time']:.2f} sec")

        # Aggregate results
        aggregated = self.evaluator.aggregate_multiple_runs(run_results)
        aggregated['convergence_histories'] = convergence_histories
        aggregated['algorithm_name'] = algorithm_name

        # Store in all_results
        self.all_results[algorithm_name] = aggregated

        print(f"\n{'='*60}")
        print(f"{algorithm_name} Experiments Complete!")
        print(f"{'='*60}")

        return aggregated

    def compare_algorithms(
        self,
        algorithms: List[str],
        metric: str = 'accuracy'
    ) -> None:
        """Compare multiple algorithms using statistical tests."""
        print(f"\n{'='*60}")
        print(f"Statistical Comparison: {metric.upper()}")
        print(f"{'='*60}\n")

        # Prepare data for comparison
        results_dict = {}
        for algo in algorithms:
            if algo in self.all_results:
                results_dict[algo] = self.all_results[algo][metric]['values']

        if len(results_dict) < 2:
            print("Need at least 2 algorithms for comparison")
            return

        # Friedman test
        statistic, p_value = self.evaluator.friedman_test(results_dict, metric)

        # Nemenyi post-hoc test (if significant)
        if p_value < 0.05:
            self.evaluator.nemenyi_test(results_dict, metric)

    def save_results(self, filename: str = 'experiment_results.pkl'):
        """Save all results to file."""
        filepath = self.results_dir / filename
        with open(filepath, 'wb') as f:
            pickle.dump(self.all_results, f)
        print(f"\nResults saved to {filepath}")

    def load_results(self, filename: str = 'experiment_results.pkl'):
        """Load results from file."""
        filepath = self.results_dir / filename
        with open(filepath, 'rb') as f:
            self.all_results = pickle.load(f)
        print(f"\nResults loaded from {filepath}")

In [ ]:
# Force reload of modules
import sys
import importlib

# Remove cached modules
modules_to_reload = [
    'experiment_manager',
    'evaluation',
    'ann_model',
    'fpa_optimizer',
    'sda_optimizer',
    'hybrid_optimizer',
    'pca_optimizer',
    'sgd_optimizer'
]

for module_name in modules_to_reload:
    if module_name in sys.modules:
        del sys.modules[module_name]
        print(f"✓ Removed {module_name} from cache")

print("\n✅ Modules cache cleared! Now re-run Cell #16 to reload.")


## 3. Dataset Preparation

Download and prepare the Chest X-ray Pneumonia dataset from Kaggle.

In [ ]:
# Setup Kaggle API credentials
import json
from pathlib import Path

# 1. Write kaggle.json content directly
kaggle_json_content = '{"username":"USERNAME","key":"KEY"}'
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
kaggle_file = kaggle_dir / 'kaggle.json'

with open(kaggle_file, 'w') as f:
    f.write(kaggle_json_content)

# 2. Set permissions
!chmod 600 ~/.kaggle/kaggle.json
print("✓ Kaggle API credentials set successfully.")

# 3. Download dataset
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/hybrid_fpa_sda/data/
!unzip -q /content/hybrid_fpa_sda/data/chest-xray-pneumonia.zip -d /content/hybrid_fpa_sda/data/

# 4. Verify dataset structure
!ls -R /content/hybrid_fpa_sda/data/chest_xray/

## 4. Import Modules and Initialize

In [ ]:
import sys
sys.path.append('/content/hybrid_fpa_sda/src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
from sklearn.model_selection import train_test_split # Added for Cell 5

# Import custom modules
from base_optimizer import BaseOptimizer
from fpa_optimizer import FlowerPollinationAlgorithm
from sda_optimizer import SeedDispersalAlgorithm
from hybrid_optimizer import HybridFPASDA
from ann_model import MLP
from data_handler import MedicalImageDataHandler
from evaluation import ModelEvaluator
from experiment_manager import ExperimentManager

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All modules imported successfully")
print(f"✓ Random seed set to {RANDOM_SEED}")

## 5. Load and Preprocess Data

Load the chest X-ray images and apply preprocessing pipeline.

In [ ]:
# Initialize data handler
data_handler = MedicalImageDataHandler(
    image_size=(150, 150),
    normalize=True,
    seed=RANDOM_SEED
)

# Load training data (use max_samples for quick testing)
print("Loading training data...")
X_train_full, y_train_full = data_handler.load_images_from_directory(
    '/content/hybrid_fpa_sda/data/chest_xray/train',
    #max_samples=1000  # Uncomment for quick testing
)

# Load test data
print("\nLoading test data...")
X_test, y_test = data_handler.load_images_from_directory(
    '/content/hybrid_fpa_sda/data/chest_xray/test',
    #max_samples=200  # Uncomment for quick testing
)

# Split training data into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,  # 20% for validation
    stratify=y_train_full,
    random_state=RANDOM_SEED
)

# Store in data_handler (for consistency)
data_handler.X_train = X_train
data_handler.y_train = y_train
data_handler.X_val = X_val
data_handler.y_val = y_val
data_handler.X_test = X_test
data_handler.y_test = y_test

print(f"\nData split complete:")
print(f"  Training set: {len(X_train)} samples")
print(f"  Validation set: {len(X_val)} samples")
print(f"  Test set: {len(X_test)} samples")

# Compute class weights
class_weights = data_handler.compute_class_weights(y_train)

## 5.1 Feature Extraction: Principal Component Analysis (PCA)

**NOTE**: PCA is applied here to reduce the high dimensionality of the image data (22,500 features) to a manageable size (500 features) for the metaheuristic algorithms. This is crucial for obtaining meaningful results.

In [ ]:
# Feature Extraction using Principal Component Analysis (PCA)
from sklearn.decomposition import PCA
import numpy as np

# 1. Define the new dimension
N_COMPONENTS = 1000

# 2. Initialize PCA
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_SEED)

# 3. Fit PCA on the training data (X_train)
print(f"Fitting PCA on {X_train.shape[0]} samples with {X_train.shape[1]} features...")
pca.fit(X_train)

# 4. Transform all datasets
X_train = pca.transform(X_train)
X_val = pca.transform(X_val)
X_test = pca.transform(X_test)

# 5. Update the model input size
NEW_INPUT_SIZE = N_COMPONENTS

print("\n--- PCA Results ---")
print(f"  Original features: {pca.n_features_in_}")
print(f"  Reduced features: {pca.n_components_}")
print(f"  Variance explained: {pca.explained_variance_ratio_.sum():.2%}")
print(f"  New X_train shape: {X_train.shape}")
print(f"  New X_val shape: {X_val.shape}")
print(f"  New X_test shape: {X_test.shape}")
print(f"\n✓ Data successfully reduced to {NEW_INPUT_SIZE} features.")


## 6. Visualize Sample Images (Using Original Data)

In [ ]:
# Reshape function for visualization
def reshape_for_display(data_flat):
    return data_flat.reshape(150, 150)

# Load a small sample of the original data for visualization
X_display = X_train_full[:8]
y_display = y_train_full[:8]

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
axes = axes.ravel()

for i in range(8):
    img = reshape_for_display(X_display[i])
    label = data_handler.class_names[y_display[i]]

    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f'Label: {label}', fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Sample Chest X-ray Images (Original Data)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Initialize Neural Network

In [ ]:
# Initialize a temporary model to get the total number of parameters
# Neural Network Architecture
HIDDEN_SIZES = [128, 64, 32]  # Use [128, 64, 32] for final paper , [64, 32] for testing the code
temp_model = MLP(input_size=NEW_INPUT_SIZE,hidden_sizes=HIDDEN_SIZES, seed=RANDOM_SEED)
INPUT_SIZE = temp_model.input_size
TOTAL_PARAMS = temp_model.total_params

print(f"Input Feature Dimension: {INPUT_SIZE}")
print(f"Total Trainable Parameters: {TOTAL_PARAMS:,}")
print(f"Layer Sizes: {temp_model.layer_sizes}")
print(f"Hidden Layer Sizes: {HIDDEN_SIZES}")

# Initialize Experiment Manager
experiment_manager = ExperimentManager(
    X_train, y_train, X_val, y_val, X_test, y_test,
    class_weights=class_weights,
    results_dir='./results'
)

print("\n✓ Experiment Manager Initialized")

## 8. Define Hyperparameters

In [ ]:
# Optimization Parameters
N_RUNS = 50         # Number of independent runs (Use 30-100 for final paper) 3 for testing the code
MAX_ITER = 150       # Maximum iterations/epochs (Use 100-200 for final paper) 3 for testing the code
POP_SIZE = 45       # Population size (Use 40-50 for final paper) 3 for testing the code
TRANSITION_POINT = 75 # Transition point for Hybrid (Use 50% of MAX_ITER)
# Set Hidden_Sizes in Step 7


print(f"Optimization Runs (N_RUNS): {N_RUNS}")
print(f"Max Iterations (MAX_ITER): {MAX_ITER}")
print(f"Population Size (POP_SIZE): {POP_SIZE}")
print(f"Hybrid Transition Point: {TRANSITION_POINT}")

## 9. Run Experiments

### 9.2 Experiment 1: Hybrid FPA-SDA (Novel)

In [ ]:

# Record the start time
start_time = datetime.now()
print(f"Execution started at: {start_time}")

# Run Hybrid FPA-SDA experiments
hybrid_results = experiment_manager.run_experiment(
    algorithm_name='Hybrid',
    n_runs=N_RUNS,
    pop_size=POP_SIZE,
    max_iter=MAX_ITER,
    transition_point=TRANSITION_POINT,
    bounds=(-1.0, 1.0),
    input_size=INPUT_SIZE,
    hidden_sizes=HIDDEN_SIZES
    # Removed seed=RANDOM_SEED as it's handled internally by run_experiment
)

# Record the end time
end_time = datetime.now()
print(f"Execution ended at: {end_time}")

# Calculate and print the duration
duration = end_time - start_time
print(f"Execution duration: {duration}")

# Quick backup after Hybrid experiment
auto_backup_results()
print("✓ Hybrid results backed up to Google Drive")

### 9.3 Experiment 2: FPA (Standalone)

In [ ]:
# Record the start time
start_time = datetime.now()
print(f"Execution started at: {start_time}")

# Run FPA experiments
fpa_results = experiment_manager.run_experiment(
    algorithm_name='FPA',
    n_runs=N_RUNS,
    pop_size=POP_SIZE,
    max_iter=MAX_ITER,
    switch_probability=0.8,
    bounds=(-1.0, 1.0),
    input_size=INPUT_SIZE,
    hidden_sizes=HIDDEN_SIZES
    # Removed seed=RANDOM_SEED
)

# Record the end time
end_time = datetime.now()
print(f"Execution ended at: {end_time}")

# Calculate and print the duration
duration = end_time - start_time
print(f"Execution duration: {duration}")

# Quick backup after FPA experiment
auto_backup_results()
print("✓ FPA results backed up to Google Drive")

### 9.4 Experiment 3: SDA (Standalone)

In [ ]:
# Record the start time
start_time = datetime.now()
print(f"Execution started at: {start_time}")

# Run SDA experiments
sda_results = experiment_manager.run_experiment(
    algorithm_name='SDA',
    n_runs=N_RUNS,
    pop_size=POP_SIZE,
    max_iter=MAX_ITER,
    dispersal_probs=(0.7, 0.15, 0.15),
    bounds=(-1.0, 1.0),
    input_size=INPUT_SIZE,
    hidden_sizes=HIDDEN_SIZES
    # Removed seed=RANDOM_SEED
)

# Record the end time
end_time = datetime.now()
print(f"Execution ended at: {end_time}")

# Calculate and print the duration
duration = end_time - start_time
print(f"Execution duration: {duration}")

# Quick backup after SDA experiment
auto_backup_results()
print("✓ SDA results backed up to Google Drive")

### 9.5 Experiment 4: PCA + Random Search (Baseline)

**Note**: This baseline uses PCA for dimensionality reduction followed by random search optimization. It demonstrates the effectiveness of feature reduction techniques.

In [ ]:
# Record the start time
import time
pca_start_time = time.time()

# Run PCA experiments
pca_results = experiment_manager.run_experiment(
    algorithm_name='PCA',
    n_runs=N_RUNS,
    pop_size=POP_SIZE,
    max_iter=MAX_ITER,
    bounds=(-1.0, 1.0),           # ← Fixed Value
    n_components=100,
    seed=None,                     # ← None
    input_size=INPUT_SIZE,
    hidden_sizes=HIDDEN_SIZES
)

pca_end_time = time.time()
pca_total_time = pca_end_time - pca_start_time

print(f"\n{'='*60}")
print(f"PCA Experiments Total Time: {pca_total_time:.2f} seconds")
print(f"{'='*60}")

# Quick backup after PCA experiment
auto_backup_results()
print("✓ PCA results backed up to Google Drive")

### 9.6 Experiment 5: SGD with Momentum (Baseline)

**Note**: This baseline uses Stochastic Gradient Descent with momentum and numerical gradient computation. It represents traditional gradient-based optimization.

In [ ]:
# Record the start time
import time
sgd_start_time = time.time()

# Run SGD experiments
sgd_results = experiment_manager.run_experiment(
    algorithm_name='SGD',
    n_runs=N_RUNS,
    pop_size=1,                    # SGD uses single solution
    max_iter=MAX_ITER,
    bounds=(-1.0, 1.0),           # Fixed Value
    learning_rate=0.001,
    momentum=0.9,
    epsilon=1e-7,
    seed=None,                     # ← None
    input_size=INPUT_SIZE,
    hidden_sizes=HIDDEN_SIZES
)

sgd_end_time = time.time()
sgd_total_time = sgd_end_time - sgd_start_time

print(f"\n{'='*60}")
print(f"SGD Experiments Total Time: {sgd_total_time:.2f} seconds")
print(f"{'='*60}")

# Quick backup after SGD experiment
auto_backup_results()
print("✓ SGD results backed up to Google Drive")

## 10. Results and Analysis

In [ ]:
# Test all algorithms
for algo in ['Hybrid', 'FPA', 'SDA', 'PCA', 'SGD']:
    if algo in experiment_manager.all_results:
        if 'training_time' in experiment_manager.all_results[algo]:
            print(f"✅ {algo}: training_time tracked")
        else:
            print(f"❌ {algo}: training_time NOT tracked")


In [ ]:
# Display aggregated results
print("\n" + "#"*60)
print("# Aggregated Results Summary")
print("#"*60)

for algo_name, results in experiment_manager.all_results.items():
    print(f"\n{algo_name}:\n" + "-"*60)
    for metric, stats in results.items():
        if metric not in ['convergence_histories', 'algorithm_name', 'values']:
            print(f"  {metric:15s}: {stats['mean']:.4f} ± {stats['std']:.4f} [{stats['min']:.4f}, {stats['max']:.4f}]")

# ============================================================================
# TRAINING TIME COMPARISON (NEW!)
# ============================================================================
print("\n" + "="*70)
print("TRAINING TIME COMPARISON")
print("="*70)

# Prepare data for training time comparison
training_time_data = []
for algo_name, results in experiment_manager.all_results.items():
    if 'training_time' in results:
        training_time_data.append({
            'Algorithm': algo_name,
            'Mean (sec)': results['training_time']['mean'],
            'Std (sec)': results['training_time']['std'],
            'Min (sec)': results['training_time']['min'],
            'Max (sec)': results['training_time']['max']
        })

# Sort by mean training time
training_time_data.sort(key=lambda x: x['Mean (sec)'])

# Display training time table
print(f"\n{'Algorithm':<15} {'Mean (sec)':<12} {'Std (sec)':<12} {'Min (sec)':<12} {'Max (sec)':<12}")
print("-" * 70)
for data in training_time_data:
    print(f"{data['Algorithm']:<15} {data['Mean (sec)']:<12.2f} {data['Std (sec)']:<12.2f} "
          f"{data['Min (sec)']:<12.2f} {data['Max (sec)']:<12.2f}")

# Identify fastest and slowest
if training_time_data:
    fastest = training_time_data[0]
    slowest = training_time_data[-1]
    print(f"\n⚡ Fastest: {fastest['Algorithm']} ({fastest['Mean (sec)']:.2f} ± {fastest['Std (sec)']:.2f} sec)")
    print(f"🐌 Slowest: {slowest['Algorithm']} ({slowest['Mean (sec)']:.2f} ± {slowest['Std (sec)']:.2f} sec)")
    speedup = slowest['Mean (sec)'] / fastest['Mean (sec)']
    print(f"📊 Speedup: {speedup:.2f}x faster")

# ============================================================================
# PERFORMANCE vs TRAINING TIME ANALYSIS (NEW!)
# ============================================================================
print("\n" + "="*70)
print("PERFORMANCE vs TRAINING TIME TRADE-OFF")
print("="*70)

# Create comparison table
comparison_data = []
for algo_name, results in experiment_manager.all_results.items():
    if 'accuracy' in results and 'training_time' in results:
        comparison_data.append({
            'Algorithm': algo_name,
            'Accuracy': results['accuracy']['mean'],
            'F1-Score': results['f1_score']['mean'],
            'MCC': results['mcc']['mean'],
            'Time (sec)': results['training_time']['mean']
        })

# Sort by accuracy (descending)
comparison_data.sort(key=lambda x: x['Accuracy'], reverse=True)

print(f"\n{'Algorithm':<15} {'Accuracy':<12} {'F1-Score':<12} {'MCC':<12} {'Time (sec)':<12}")
print("-" * 70)
for data in comparison_data:
    print(f"{data['Algorithm']:<15} {data['Accuracy']:<12.4f} {data['F1-Score']:<12.4f} "
          f"{data['MCC']:<12.4f} {data['Time (sec)']:<12.2f}")

# Identify best performer
if comparison_data:
    best = comparison_data[0]
    print(f"\n🏆 Best Performance: {best['Algorithm']} "
          f"(Accuracy: {best['Accuracy']:.4f}, Time: {best['Time (sec)']:.2f} sec)")

# ============================================================================
# VISUALIZATION: CONVERGENCE CURVES
# ============================================================================
print("\n" + "="*70)
print("CONVERGENCE ANALYSIS")
print("="*70)

# Plot convergence curves
convergence_data = {
    algo: np.mean(results['convergence_histories'], axis=0)
    for algo, results in experiment_manager.all_results.items()
}

experiment_manager.evaluator.plot_convergence_curves(
    convergence_data,
    title="Mean Convergence Curves (Loss)"
)

# ============================================================================
# VISUALIZATION: METRICS COMPARISON
# ============================================================================
print("\n" + "="*70)
print("METRICS COMPARISON")
print("="*70)

# Plot metrics comparison
experiment_manager.evaluator.plot_metrics_comparison(
    experiment_manager.all_results,
    metrics=['accuracy', 'f1_score', 'mcc']
)

# ============================================================================
# VISUALIZATION: TRAINING TIME BAR CHART (NEW!)
# ============================================================================
print("\n" + "="*70)
print("TRAINING TIME VISUALIZATION")
print("="*70)

import matplotlib.pyplot as plt

# Prepare data for bar chart
algorithms = [d['Algorithm'] for d in training_time_data]
mean_times = [d['Mean (sec)'] for d in training_time_data]
std_times = [d['Std (sec)'] for d in training_time_data]

# Create bar chart
plt.figure(figsize=(12, 6))
bars = plt.bar(algorithms, mean_times, yerr=std_times, capsize=5, alpha=0.7,
               color=['#2ecc71' if i == 0 else '#e74c3c' if i == len(algorithms)-1 else '#3498db'
                      for i in range(len(algorithms))])

plt.xlabel('Algorithm', fontsize=12, fontweight='bold')
plt.ylabel('Training Time (seconds)', fontsize=12, fontweight='bold')
plt.title('Training Time Comparison Across Algorithms', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()

# Add value labels on bars
for i, (bar, mean_time, std_time) in enumerate(zip(bars, mean_times, std_times)):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{mean_time:.1f}±{std_time:.1f}',
             ha='center', va='bottom', fontsize=9)

plt.show()

print("\n✓ Analysis Complete")
print("="*70)

## 11. Statistical Analysis

Perform Friedman and Nemenyi post-hoc tests to determine statistical significance.

In [ ]:
metric_to_test = 'mcc'

# Prepare data for statistical tests
statistical_data = {
    algo: results[metric_to_test]['values']
    for algo, results in experiment_manager.all_results.items()
}

print(f"\n{'#'*60}")
print(f"# Statistical Analysis on {metric_to_test.upper()}")
print(f"{'#'*60}")

# Friedman Test
stat, p_val = experiment_manager.evaluator.friedman_test(
    statistical_data,
    metric_name=metric_to_test
)

# Nemenyi Post-hoc Test (only if Friedman is significant)
if p_val < 0.05:
    experiment_manager.evaluator.nemenyi_test(
        statistical_data,
        metric_name=metric_to_test
    )
else:
    print("\nNemenyi test skipped as Friedman test was not significant.")

# Full backup at the end
print("\n" + "="*70)
print("FINAL BACKUP TO GOOGLE DRIVE")
print("="*70)
backup_to_drive(exclude_dataset=True)
print("\n✓ All experiments completed and backed up!")
print(f"✓ Results saved to: {DRIVE_PROJECT_PATH}")

## 12. Conclusion

The results of the Hybrid FPA-SDA algorithm, compared against FPA, SDA, and the PCA, SGD baseline, are presented above.